# LLM Agent Workflows
 
 RAG study assistant built with LangChain and a movie recommendation workflow built with LangGraph.


In [73]:
import os
from typing import List

from langchain_openai import ChatOpenAI

In [ ]:
key = os.environ.get("OPENROUTER_API_KEY", "")


In [75]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.base import RunnableSerializable
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

In [76]:
llm = ChatOpenAI(
    api_key=key,
    base_url="https://openrouter.ai/api/v1",
    model="qwen/qwen3.6-plus-preview:free"
)

response = llm.invoke("Привет! Как твои дела?").content
print(response)

Привет! Спасибо, что спросил. 😊 У меня всё отлично, я готов помогать и отвечать на твои вопросы. А как твои дела? Чем могу быть полезен сегодня? 💬✨


**ЗАДАНИЕ 1. LANG-CHAIN MODEL. Помощник подготовки к экзаменам**

Здесь я решил реализовать базовый пример с Rag по поиску нужной информации из источников. модель так же может создавать вопросы по шаблону и объяснять сложные термины.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

loader = PyPDFLoader("Все презентации МАБП.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"},
)


vectorstore = FAISS.from_documents(splits, embeddings)

retriever = vectorstore.as_retriever()

In [ ]:
@tool
def search_docs(query: str) -> str:
    """Ищет информацию в PDF документах"""
    docs = vectorstore.similarity_search(query, k=5)
    return "\n\n".join([d.page_content for d in docs])


@tool
def simplify(text: str) -> str:
    """Упрощает сложный текст"""
    return f"Объясни простыми словами эти термины: {text}"


@tool
def generate_questions(text: str) -> str:
    """Генерирует вопросы по теме"""
    return f"Сгенерируй вопросы для проверки знаний: {text}. Сначала напиши все вопросы, а потом ответы на него, с большим отступом, чтобы они не бросались в глаза."

In [79]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Ты ассистент по экзаменам. "
     "Отвечай по документам, упрощай материал и задавай вопросы. "
     "Используй инструменты, если нужно."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [80]:
tools = [search_docs, simplify, generate_questions]

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | agent_prompt
    | llm.bind_tools(tools, tool_choice="auto")
)

In [81]:
from langchain_core.messages import HumanMessage, AIMessage

class CustomAgentExecutor:
    def __init__(self, agent, tools):
        self.agent = agent
        self.tools = {tool.name: tool for tool in tools}
        self.chat_history = []

    def invoke(self, user_input: str):
        response = self.agent.invoke({
            "input": user_input,
            "chat_history": self.chat_history
        })

        if response.tool_calls:
            call = response.tool_calls[0]
            tool_name = call["name"]
            tool_args = call["args"]

            tool_result = self.tools[tool_name].invoke(tool_args)

            final = self.agent.invoke({
                "input": f"Используй результат инструмента:\n{tool_result}",
                "chat_history": self.chat_history
            })

            answer = final.content
        else:
            answer = response.content

        self.chat_history.append(HumanMessage(content=user_input))
        self.chat_history.append(AIMessage(content=answer))

        return answer
    
executor = CustomAgentExecutor(agent, tools)

In [82]:
result = executor.invoke('привет. что ты делаешь?')
print(f"Результат: {result}")

Результат: Привет! Я ассистент для подготовки к экзаменам. Помогаю эффективно учиться тремя основными способами:
📄 **Ищу информацию** в твоих PDF-документах по конкретным темам или вопросам
🔍 **Упрощаю сложный материал** — переписываю запутанные формулировки простым языком
❓ **Генерирую вопросы** для самопроверки, чтобы ты мог отработать материал перед экзаменом

Скажи, к какому предмету или экзамену готовишься? Есть ли у тебя документы/конспекты, с которыми нужно поработать, или конкретная тема, которую стоит разобрать?


[(Document(id='e0ff4111-18ae-4507-94b7-9f205907279c', metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-12-23T10:18:13+00:00', 'source': 'Все презентации МАБП.pdf', 'total_pages': 472, 'page': 408, 'page_label': '409'}, page_content='Стратегическая карта электронного правительства'),
  np.float32(0.29616964)),
 (Document(id='3278ef23-af06-4b17-9ad5-ef877b6bc004', metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-12-23T10:18:13+00:00', 'source': 'Все презентации МАБП.pdf', 'total_pages': 472, 'page': 421, 'page_label': '422'}, page_content='ЭТАПЫ ЦИФРОВОЙ ТРАНСФОРМАЦИИ'),
  np.float32(0.3314013))]

In [84]:
print(f"Результат: {executor.invoke('привет. расскажи мне какие бывают типы событий в МАБП?')}")

Результат: Привет! В курсе «Моделирование и анализ бизнес-процессов» (МАБП) **событие** — это дискретный факт, который влияет на ход выполнения процесса: запускает его, меняет маршрут или завершает. В подавляющем большинстве вузов классификация строится на стандарте **BPMN 2.0**. Вот экзаменационно-ориентированная разбивка:

### 🔹 1. По положению в потоке управления
| Тип | Роль | Визуал (BPMN) |
|-----|---------------------|
| **Начальное (Start)** | Запускает процесс или подпроцесс. Не может иметь входящих потоков. | Окружность с тонкой одинарной рамкой |
| **Промежуточное (Intermediate)** | Происходит между стартом и финишем. Ждёт триггер или генерирует воздействие. | Окружность с двойной рамкой |
| **Конечное (End)** | Завершает ветку или весь процесс. Не может иметь исходящих потоков (кроме компенсации/эскалации). | Окружность с жирной тройной рамкой |

### 🔹 2. По типу триггера / источнику (самая частая тема на экзамене)
| Тип события | Что делает | Пример |
|-------------|------

In [85]:
print(f"Результат: {executor.invoke('создай вопросы по только что пройденной теме')}")

Результат: Вот подборка вопросов разного уровня сложности, составленная строго по классификации событий МАБП (BPMN 2.0). Используй их для самопроверки, устного опроса или написания тестов.

### 🟢 Базовый уровень (термины и обозначения)
1. Перечислите три типа событий по их положению в потоке управления. Какие из них **всегда** являются `Catch`, а какие **всегда** `Throw`?
2. Что означает двойная рамка вокруг события в нотации BPMN 2.0? Как визуально отличаются начальное, промежуточное и конечное события?
3. В чём принципиальная разница между режимами `Catch` (перехват) и `Throw` (генерация)? Приведите по одному примеру для каждого режима.

### 🟡 Средний уровень (сравнение и логика)
4. Сравните события `Error` и `Escalation`. В каком случае нарушение приведёт к прерыванию основного потока, а в каком — только к уведомлению без остановки процесса?
5. Чем `Signal` отличается от `Message` с точки зрения адресата и области видимости? Когда использовать широкое оповещение, а когда точечную пе

**ЗАДАНИЕ 2. LANG-GRAF MODEL. Подборка фильмов через кинопоиск**

Этот агент по запросу анализирует предпочтения пользователя -> парсит их на фильтры -> по апи кинопоиска ищутся нужные фильмы -> ии анализирует данные и выводит топ 3 лучших фильма, описывая их. В итоге получился прикольный советчик фильмов. Мне очень понравился результат, так как модель выдает действительно интересную подборку, которую я сам вряд ли бы сам нашел, вручную вбивая фильтры на каком нибудь сайте.

In [ ]:
kp_key = os.environ.get("KINOPOISK_API_KEY", "")


In [ ]:
import requests
from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
import operator
import json


class AgentState(TypedDict):
    messages: Annotated[List[Dict], operator.add]  
    user_preferences: Dict[str, Any]               
    api_filters: Dict[str, Any]                     
    api_response: Dict[str, Any]                    
    shortlist: List[Dict]                           
    final_recommendation: str                       
    errors: List[str]

def search_movies(filters: Dict[str, Any]) -> Dict[str, Any]:
    base_url = "https://api.poiskkino.dev/v1.4/movie"
    headers = {"X-API-KEY": kp_key}
    try:
        response = requests.get(base_url, headers=headers, params=filters)
        return response.json()
    except Exception as e:
        return {"error": str(e)}
    

def parse_preferences(state: AgentState) -> AgentState:
    user_input = state["messages"][-1].content
    
    prompt = f"""
    Извлеки из запроса пользователя параметры для поиска фильмов.
    Формат ответа — только валидный JSON без дополнительного текста:
    {{
        "year": "диапазон годов (2020-2024) или конкретный год",
        "rating.kp": "диапазон рейтинга кинопоиска (7.1-10)",
        "rating.imdb": "диапазон рейтинга IMDB (7.1-10)",
        "genres_include": ["жанр1", "жанр2"],
        "genres_exclude": ["жанр3"],
        "is_series": true/false/null,
        "country_include": ["страна1", "страна2"],
        "country_exclude": ["страна3"],
        "votes.kp": "диапазон количества голосов на кинопоиске (1000-9999999 если не указано пользователем)",
        'limit': 20
    }}
    Запрос: "{user_input}"
    """
    response = llm.invoke([SystemMessage(content=prompt)])
    try:
        state["user_preferences"] = json.loads(response.content.strip())
    except Exception as e:
        state["user_preferences"] = {}
    return state

def build_api_filters(state: AgentState) -> AgentState:
    prefs = state["user_preferences"]
    filters = {}

    if prefs.get("year"):
        filters["year"] = prefs["year"]
    if prefs.get("rating.kp"):
        filters["rating.kp"] = prefs["rating.kp"]
    if prefs.get("rating.imdb"):
        filters["rating.imdb"] = prefs["rating.imdb"]

    genres = []
    for g in prefs.get("genres_include", []):
        genres.append(f"+{g}")
    for g in prefs.get("genres_exclude", []):
        genres.append(f"!{g}")
    if genres:
        filters["genres.name"] = genres
    
    if prefs.get("is_series") is not None:
        filters["isSeries"] = str(prefs["is_series"]).lower()
    
    countries = []
    for c in prefs.get("country_include", []):
        countries.append(f"+{c}")
    for c in prefs.get("country_exclude", []):
        countries.append(f"!{c}")
    if countries:
        filters["countries.name"] = countries
    
    if prefs.get("votes.kp"):
        filters["votes.kp"] = prefs["votes.kp"]
    if prefs.get("limit"):
        filters["limit"] = str(prefs["limit"])

    state["api_filters"] = filters
    return state

def call_api(state: AgentState) -> AgentState:
    result = search_movies(state["api_filters"])
    if "error" in result:
        state["errors"].append(result["error"])
    else:
        state["api_response"] = result
    return state

def select_best_movies(state: AgentState) -> AgentState:
    movies = state["api_response"].get("docs", [])[:10]
    
    prompt = f"""
    Пользователь ищет фильм с такими предпочтениями: {state['user_preferences']}
    
    Вот список подходящих фильмов:
    {json.dumps(movies, ensure_ascii=False)}
    
    Выбери ТОП-3 лучших варианта, максимально соответствующих запросу.
    Для каждого кратко напиши:
    Название и год
    Почему он хорошо подходит и о чем он (1-2 предложения)
    Один интересный факт о фильме или актёрах
    """
    response = llm.invoke([SystemMessage(content=prompt)])
    state["final_recommendation"] = response.content
    return state

def create_agent():
    graph = StateGraph(AgentState)
    graph.add_node("parse_prefs", parse_preferences)
    graph.add_node("build_filters", build_api_filters)
    graph.add_node("call_api", call_api)
    graph.add_node("select_movies", select_best_movies)
    
    graph.set_entry_point("parse_prefs")
    graph.add_edge("parse_prefs", "build_filters")
    graph.add_edge("build_filters", "call_api")
    graph.add_edge("call_api", "select_movies")
    graph.add_edge("select_movies", END)
    
    return graph.compile()
agent = create_agent()
agent

In [102]:
response = agent.invoke({"messages": [HumanMessage(content="привет. посоветуй фильмы за последние 3 года в жанре фантастика с высоким рейтингом.")]})
print(response["final_recommendation"])

Вот ТОП-3 фильма, которые лучше всего соответствуют вашим критериям (2022–2024, КП ≥ 7.0, жанр «фантастика», полный метр):

1. **Человек-паук: Паутина вселенных (2023)**
- **Почему подходит и о чём:** Рейтинг 8.34. Эталонный научно-фантастический мультфильм о мультивселенной: Майлз Моралес прыгает между параллельными реальностями, чтобы остановить злодея Пятно и понять, что значит быть героем для тех, кого любишь.
- **Факт:** Аниматоры разработали специальный рендерер, имитирующий полиграфические «ошибки» комиксов (сдвиг контуров, точки растра и смещение красок), для чего потребовалось создать более 240 уникальных визуальных палитр под каждое измерение.

2. **Дикий робот (2024)**
- **Почему подходит и о чём:** Рейтинг 8.35. Глубокая фантастическая история о сервисном роботе РОЗЗ, который после кораблекрушения оказывается на диком острове и, обучаясь языку животных, постепенно обретает инстинкты и материнскую привязанность к сироте-гусёнку.
- **Факт:** Лупита Нионго записывала реплики д

Сегодня LLM работает намного медленне чем вчера, скорее всего из-за нагрузок на сервере. Так что для быстроты можно взять другую модель.

In [105]:
response = agent.invoke({"messages": [HumanMessage(content="привет. посоветуй какие то интересные американские сериалы 2010-2020 годов в жанре криминал с рейтингом больше 6.")]})
print(response["final_recommendation"])


Вот ТОП-3 сериала, которые максимально точно соответствуют вашим фильтрам и выделяются качеством, рейтингом и relevancy к жанру:

**1. Власть в ночном городе. Книга вторая: Призрак (2020)**
- **Почему подходит и о чём:** Лидер по рейтингу в подборке (КП: 7.86), чистокровная криминальная драма от США. Сериал продолжает вселенную «Power», фокусируясь на Тарике — сыне наркобарона, который пытается разорвать порочный круг преступного мира, одновременно учась в престижном университете и сводя счёты с врагами семьи.
- **Интересный факт:** Исполнитель главной роли Майкл Рэйни-младший изначально играл Тарика как второстепенного персонажа в оригинальном сериале, но настолько запомнился зрителям и критикам своим мрачным взрослением, что шоураннеры полностью переписали лор спин-оффа под его персонажа.

**2. 24 часа: Проживи ещё один день (2014)**
- **Почему подходит и о чём:** Классика напряжённого криминального триллера (КП: 7.79) в формате «реального времени». Действие перенесено в Лондон, где 

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="привет. посоветуй какие то интересные малоизвестные высокорейтинговые драмы не из США.")]})
print(response["final_recommendation"])


---------------
Вот ТОП-3 варианта, которые точно проходят по вашим фильтрам (КП 7.5–10, голосов 1000–10000, жанр «драма», не США) и обладают наибольшей художественной ценностью:

**1. Евгений Гришковец: Предисловие (2019)**
- Почему идеально подходит и о чём: Рейтинг 8.89 при почти 9.5 тыс. голосов подтверждает высочайшее зрительское признание в рамках чистой драмы. Это камерный моноспектакль о том, как из бытовых воспоминаний рождается литература, где автор искренне рассуждает о творчестве, времени и любви к жизни без сценарных клише.
- Интересный факт: Проект начинался как живой театральный перформанс в Москве, а для экранизации режиссёр намеренно отказался от монтажа и декораций, оставив только актёра, свет и «воздух» сцены, чтобы максимально сохранить эффект личного исповедального разговора.

**2. Франкенштейн: Камбербэтч (2011)**
- Почему идеально подходит и о чём: КП 8.72 и жанровая комбинация «драма/фантастика» отлично укладываются в запрос. Это глубокая психологическая притча 